# 10 ? Final Demand-Forecasting Models (Consolidated)

This notebook is the single, organized view of the project's **final selected model per series** and the resulting **24-month demand forecast** (2026-01 to 2027-12). It is a *presentation / consolidation* notebook: it **reads the productionized script outputs as the source of truth** and presents them ? it does **not** re-fit models or re-run model selection, so it can never drift from what the production script (`scripts/05_modeling_with_cnmc.py`) actually produced.

### Final policy: non-pooled production models
Pooled regional ML models are still evaluated as an experiment, but they are **not used in the final production selected model set**. The 2025 period is treated as a **validation / acceptance period**, not as a pristine final test set.

| Series | Final model | Pooled? | 2025 validation MAPE |
|---|---|---|---|
| Nacional | SARIMA | no | 29.0% |
| Madrid | Logistic curve | no | 73.6% |
| **Catalu?a** | **SARIMA** | **no** | **47.2%** |
| Andaluc?a | Logistic curve | no | 48.4% |
| Valencia | Gompertz curve | no | 34.2% |

> The pooled Catalu?a model remains useful as a sensitivity case, but the final deliverable uses the best non-pooled Catalu?a model.


## 0. Setup — load the productionized Phase 2 outputs

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
REPO_ROOT    = NOTEBOOK_DIR.parent
OUT          = REPO_ROOT / 'data' / 'outputs'
FEAT         = REPO_ROOT / 'data' / 'features'

TARGETS = ['Nacional', 'Madrid', 'Cataluña', 'Andalucía', 'Valencia']
COLORS  = {'Nacional': '#FF6B35', 'Madrid': '#004E89', 'Cataluña': '#1A936F',
           'Andalucía': '#C84B31', 'Valencia': '#8E44AD'}

final    = pd.read_csv(OUT / 'metricas_final_seleccionado.csv')
accept   = pd.read_csv(OUT / 'phase2_model_acceptance.csv')
pool_exp = pd.read_csv(OUT / 'phase2_pooling_experiment_metrics.csv')
preds    = pd.read_csv(OUT / 'predicciones_test_2025.csv')
forecast = pd.read_csv(OUT / 'forecast_24m_sarima_rf_xgb.csv')
history  = pd.read_csv(FEAT / 'features_modelo_completo.csv')[['Fecha', 'Target', 'Consumo_Tm']]

SELECTED = dict(zip(final['Target'], final['Model']))
plt.rcParams['figure.dpi'] = 110
print('Final selected model per series:')
for t in TARGETS:
    print('  {:10s} -> {}'.format(t, SELECTED[t]))

## 1. Final selected models & how each was chosen

The headline selection, then the decision trail. Models are proposed by the recursive multi-step walk-forward gate on 2023-2024. The 2025 period is used as a validation / acceptance period. A final no-pooling delivery policy then chooses the best non-pooled Catalu?a alternative, SARIMA, instead of the pooled Random Forest sensitivity case.


In [ ]:
# 1a. Final selected models + 2025 holdout metrics
tbl = final.copy()
tbl['Pooled'] = np.where(tbl['Model'].str.startswith('Pooled'), 'yes', 'no')
tbl = tbl[['Target', 'Model', 'Pooled', 'MAE', 'RMSE', 'MAPE', 'R2']]
print('FINAL SELECTED MODELS (2025 validation period):')
print(tbl.to_string(index=False))
avg = final['MAPE'].mean()
print()
print('Average selected 2025 validation MAPE: {:.1f}%  (was 94.6% before Phase 2; final set is non-pooled)'.format(avg))

# 1b. Selection decision per series (Phase 1 vs Phase 2 proposal vs final)
print()
print('SELECTION DECISION (phase2_model_acceptance.csv):')
acc_cols = [c for c in ['Target', 'Phase1_Model', 'Phase1_MAPE', 'Phase2_Proposed_Model',
                        'Phase2_Proposed_MAPE', 'Non_Pooled_Final_Model',
                        'Non_Pooled_Final_MAPE', 'Selected_Model',
                        'Final_Selection_Source', 'Decision'] if c in accept.columns]
acc = accept[acc_cols]
print(acc.to_string(index=False))

# 1c. Pooled candidates on the 2025 holdout -> why pooling helps only Cataluña
print()
print('POOLED CANDIDATES, 2025 holdout MAPE (%):')
pe = pool_exp.pivot_table(index='Target', columns='Model', values='MAPE')
cols = [c for c in ['Pooled Ridge', 'Pooled Random Forest', 'Pooled XGBoost'] if c in pe.columns]
print(pe[cols].round(1).to_string())
print()
print('Pooled models are reported as sensitivities; final production selections are non-pooled.')

## 2. Results — 2025 holdout fit & the 24-month forecast

First, each series' selected model against the realized 2025 values. Then the deliverable: 2023-2025 history plus the 24-month forward forecast (the dotted grey line marks the train/forecast boundary at 2025-12).

In [ ]:
# 2025 holdout: selected model vs actuals
fig, axes = plt.subplots(3, 2, figsize=(15, 12)); axes = axes.ravel()
for i, t in enumerate(TARGETS):
    ax = axes[i]; m = SELECTED[t]
    d = preds[(preds['Target'] == t) & (preds['Model'] == m)].sort_values('Fecha').copy()
    d['date'] = pd.to_datetime(d['Fecha'])
    ax.plot(d['date'], d['Actual'], 'o-', color='black', lw=2, ms=4, label='Actual')
    ax.plot(d['date'], d['Pred'], 's--', color=COLORS[t], lw=2, ms=4, label=m)
    mape = float(final[final['Target'] == t]['MAPE'].iloc[0])
    ax.set_title('{} - {}  (2025 MAPE {:.1f}%)'.format(t, m, mape), fontsize=11, weight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=8); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_ylabel('Biodiesel (Tm)')
axes[-1].axis('off')
fig.suptitle('2025 holdout: selected model vs actuals', fontsize=13, weight='bold')
fig.tight_layout(); plt.show()

In [ ]:
# 24-month forecast (2026-2027): history + selected-model forecast
fig, axes = plt.subplots(3, 2, figsize=(15, 12)); axes = axes.ravel()
cut = pd.to_datetime('2025-12')
for i, t in enumerate(TARGETS):
    ax = axes[i]; m = SELECTED[t]
    h = history[history['Target'] == t].sort_values('Fecha').copy()
    fc = forecast[(forecast['Target'] == t) & (forecast['Model'] == m)].sort_values('Fecha').copy()
    h['date'] = pd.to_datetime(h['Fecha']); fc['date'] = pd.to_datetime(fc['Fecha'])
    ax.plot(h['date'], h['Consumo_Tm'], '-', color='black', lw=1.8, label='Historico (2023-25)')
    ax.plot(fc['date'], fc['Forecast'], '--', color=COLORS[t], lw=2.2, label='Forecast ({})'.format(m))
    ax.axvline(cut, color='grey', ls=':', lw=1)
    ax.set_title('{} - {}'.format(t, m), fontsize=11, weight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=8); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_ylabel('Biodiesel (Tm)')
axes[-1].axis('off')
fig.suptitle('24-month biodiesel demand forecast (2026-2027)', fontsize=13, weight='bold')
fig.tight_layout(); plt.show()

In [ ]:
# Annual forecast totals (sum of each series' SELECTED model over each year)
f = forecast.merge(pd.DataFrame(SELECTED.items(), columns=['Target', 'Model']), on=['Target', 'Model'])
f['Year'] = f['Fecha'].str[:4]
annual = f.pivot_table(index='Target', columns='Year', values='Forecast', aggfunc='sum').round(0)
annual = annual.reindex(TARGETS)
print('Forecast biodiesel demand (Tm) - annual totals of the per-series selected model:')
print(annual.to_string())

## Summary & caveats

- **What this is:** the consolidated final model set. Pooled regional models are tested and reported as sensitivities, but the production selected set is **non-pooled**. Average 2025 validation MAPE is approximately **46.5%**.
- **Honest limits (carry these into any Repsol-facing deck):**
  - Only 36 months of data (24 training); R2 is negative for every series except near-zero Nacional. Present these as **directional planning scenarios**, not precise demand commitments.
  - **Catalu?a uses SARIMA in the final non-pooled set.** It is almost tied with the pooled Random Forest on 2025 validation MAPE, but it extrapolates a growth trend rather than producing the pooled tree model's flat plateau. Flag this as a scenario-level choice.
  - The 2025 period is an acceptance / validation period because it is used in the no-regression and no-pooling final-selection decision. Do not describe it as a pristine final test set.
- **Provenance:** all numbers are read from `data/outputs/`. Source of truth = `scripts/05_modeling_with_cnmc.py`; full rationale = `PHASE2_MODELING_REPORT.md`. This notebook re-derives nothing.
